# Oxford–Harvard atlas: GIST, CLIP-ViT, and Valence-Arousal

Three compact horizontal fsaverage flatmaps using the symmetric-split Oxford–Harvard 25% maximum-probability cortical atlas at 1 mm. The same ROI overlay is the union of every qualifying parcel reached by any of the three statistical maps. Each panel has its own bottom colorbar.

In [ ]:
"""Oxford-Harvard cortical atlas variant of surface_plot.ipynb.

Run inside the WSL ``pycortex`` conda environment.  The script projects the
Oxford-Harvard maximum-probability cortical atlas and two thresholded t-maps
onto the fsaverage surface, cleans the parcel masks on the mesh, and creates
publication-ready flatmaps with matching ROI outlines.
"""

from pathlib import Path
import re

import cortex
import cortex.database
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection
from nilearn import datasets, surface
from PIL import Image
from scipy import sparse


# -----------------------------------------------------------------------------
# Paths and display settings
# -----------------------------------------------------------------------------
PROJECT_DIR = Path(
    "/mnt/n/Experimental_Data/yujunchen/projects/IAPS_Searchlight"
)
PY_CORTEX_DIR = PROJECT_DIR / "visualization/pycortex"

CLIP_TMAP = (
    PROJECT_DIR
    / "outputs/tBrainmap/tmap_clipvit_p001_fisherz_fdr001_th9.5_cluster100.nii.gz"
)
GIST_TMAP = (
    PROJECT_DIR
    / "outputs/tBrainmap/tmap_gist_norm_p001_fisherz_fdr001_th9.5_cluster100.nii.gz"
)
VA_TMAP = (
    PROJECT_DIR
    / "outputs/tBrainmap/final_results/tmap_beh60_p05_v2.nii.gz"
)

OUT_CLIP = PY_CORTEX_DIR / "surface_plot_oxford_harvard_clipvit.png"
OUT_GIST = PY_CORTEX_DIR / "surface_plot_oxford_harvard_gist.png"
OUT_VA = PY_CORTEX_DIR / "surface_plot_oxford_harvard_valence_arousal.png"
OUT_COMBINED = PY_CORTEX_DIR / "surface_plot_oxford_harvard_three_maps.png"
OUT_LATEST = PY_CORTEX_DIR / "surface_plot_oxford_harvard.png"
OUT_SUMMARY = PY_CORTEX_DIR / "surface_plot_oxford_harvard_roi_summary.csv"

ATLAS_NAME = "cort-maxprob-thr25-1mm"
TMAP_THRESHOLD = 0.01
MIN_ACTIVE_VERTICES = 25
MIN_ACTIVE_PERCENT = 0.50
SMOOTHING_STEPS = 3
SMOOTHING_CUTOFF = 0.48
# Remove the far lateral wings equally in all three panels.  This retains the
# atlas parcels reached by these maps while making a horizontal layout compact.
FLATMAP_XLIM = (-188, 195)

ROI_COLORS = [
    "#E76F51", "#F4A261", "#E9C46A", "#A7C957", "#2A9D8F",
    "#22A7B8", "#3A86C8", "#6C8CD5", "#8E7DBE", "#B07AA1",
    "#D17A9B", "#F08A8B", "#78A6A3", "#C2A878",
    "#7F8C8D", "#D4A373", "#90BE6D", "#577590", "#B56576",
    "#6D597A", "#43AA8B", "#F8961E", "#277DA1", "#BC6C25",
]


SHORT_LABELS = {
    "Frontal Pole": "FP",
    "Insular Cortex": "Insula",
    "Superior Frontal Gyrus": "SFG",
    "Middle Frontal Gyrus": "MFG",
    "Inferior Frontal Gyrus, pars triangularis": "IFGtri",
    "Inferior Frontal Gyrus, pars opercularis": "IFGop",
    "Precentral Gyrus": "Precentral",
    "Temporal Pole": "TP",
    "Superior Temporal Gyrus, anterior division": "aSTG",
    "Superior Temporal Gyrus, posterior division": "pSTG",
    "Middle Temporal Gyrus, anterior division": "aMTG",
    "Middle Temporal Gyrus, posterior division": "pMTG",
    "Middle Temporal Gyrus, temporooccipital part": "toMTG",
    "Inferior Temporal Gyrus, anterior division": "aITG",
    "Inferior Temporal Gyrus, posterior division": "pITG",
    "Inferior Temporal Gyrus, temporooccipital part": "toITG",
    "Postcentral Gyrus": "Postcentral",
    "Superior Parietal Lobule": "SPL",
    "Supramarginal Gyrus, anterior division": "aSMG",
    "Supramarginal Gyrus, posterior division": "pSMG",
    "Angular Gyrus": "AG",
    "Lateral Occipital Cortex, superior division": "sLOC",
    "Lateral Occipital Cortex, inferior division": "iLOC",
    "Intracalcarine Cortex": "ICC",
    "Frontal Medial Cortex": "FMC",
    "Juxtapositional Lobule Cortex (formerly Supplementary Motor Cortex)": "SMA",
    "Subcallosal Cortex": "Subcallosal",
    "Paracingulate Gyrus": "ParaCG",
    "Cingulate Gyrus, anterior division": "aCG",
    "Cingulate Gyrus, posterior division": "pCG",
    "Precuneous Cortex": "Precuneus",
    "Cuneal Cortex": "Cuneus",
    "Frontal Orbital Cortex": "OFC",
    "Parahippocampal Gyrus, anterior division": "aPHG",
    "Parahippocampal Gyrus, posterior division": "pPHG",
    "Lingual Gyrus": "LG",
    "Temporal Fusiform Cortex, anterior division": "aTFG",
    "Temporal Fusiform Cortex, posterior division": "pTFG",
    "Temporal Occipital Fusiform Cortex": "toFG",
    "Occipital Fusiform Gyrus": "oFG",
    "Frontal Operculum Cortex": "FrOP",
    "Central Opercular Cortex": "COp",
    "Parietal Operculum Cortex": "POp",
    "Planum Polare": "PP",
    "Heschl's Gyrus (includes H1 and H2)": "HG",
    "Planum Temporale": "PT",
    "Supracalcarine Cortex": "SCC",
    "Occipital Pole": "OP",
}


def base_region(label):
    """Remove the hemisphere prefix used by symmetric-split atlas labels."""
    return re.sub(r"^(Left|Right)\s+", "", label).strip()


def load_tmap_texture(path, fsaverage):
    """Project a volumetric t-map through the cortical ribbon."""
    left = surface.vol_to_surf(
        path,
        fsaverage.pial_left,
        inner_mesh=fsaverage.white_left,
        interpolation="linear",
        n_samples=7,
    )
    right = surface.vol_to_surf(
        path,
        fsaverage.pial_right,
        inner_mesh=fsaverage.white_right,
        interpolation="linear",
        n_samples=7,
    )
    texture = np.hstack([left, right])
    texture[np.abs(texture) < TMAP_THRESHOLD] = np.nan
    return texture


def project_atlas(atlas_img, fsaverage):
    """Project discrete atlas labels using modal sampling through the ribbon."""
    left = surface.vol_to_surf(
        atlas_img,
        fsaverage.pial_left,
        inner_mesh=fsaverage.white_left,
        interpolation="nearest_most_frequent",
        n_samples=9,
    )
    right = surface.vol_to_surf(
        atlas_img,
        fsaverage.pial_right,
        inner_mesh=fsaverage.white_right,
        interpolation="nearest_most_frequent",
        n_samples=9,
    )
    return np.nan_to_num(np.hstack([left, right]), nan=0).astype(np.int16)


def normalized_mesh_adjacency(n_vertices, triangles):
    """Sparse row-normalized adjacency used for topology-aware mask cleanup."""
    edges = np.vstack(
        [triangles[:, [0, 1]], triangles[:, [1, 2]], triangles[:, [2, 0]]]
    )
    edges = np.vstack([edges, edges[:, ::-1]])
    rows = np.hstack([edges[:, 0], np.arange(n_vertices)])
    cols = np.hstack([edges[:, 1], np.arange(n_vertices)])
    weights = np.hstack([np.ones(len(edges)), np.full(n_vertices, 2.0)])
    graph = sparse.coo_matrix(
        (weights, (rows, cols)), shape=(n_vertices, n_vertices)
    ).tocsr()
    graph.sum_duplicates()
    row_sum = np.asarray(graph.sum(axis=1)).ravel()
    return sparse.diags(1.0 / np.maximum(row_sum, 1.0)) @ graph


def smooth_mask(mask, adjacency):
    """Smooth a binary parcel on the cortical mesh without blurring the t-map."""
    score = mask.astype(np.float32)
    for _ in range(SMOOTHING_STEPS):
        score = adjacency @ score
    cleaned = score >= SMOOTHING_CUTOFF
    # Keep atlas support nearby and avoid islands created far from the parcel.
    support = (adjacency @ mask.astype(np.float32)) > 0
    return cleaned & support


def boundary_segments(mask, points, triangles):
    """Return unique mesh edges that form a parcel boundary."""
    edges = np.vstack(
        [triangles[:, [0, 1]], triangles[:, [1, 2]], triangles[:, [2, 0]]]
    )
    edge_mask = mask[edges[:, 0]] != mask[edges[:, 1]]
    edges = np.sort(edges[edge_mask], axis=1)
    if len(edges) == 0:
        return np.empty((0, 2, 2), dtype=float)
    edges = np.unique(edges, axis=0)
    return points[edges, :2]


def label_anchor(mask, points):
    """Choose an in-parcel vertex near the flat-space centroid."""
    vertex_ids = np.flatnonzero(mask)
    parcel_points = points[vertex_ids, :2]
    center = np.median(parcel_points, axis=0)
    return parcel_points[np.argmin(np.sum((parcel_points - center) ** 2, axis=1))]


def active_stats(texture, mask):
    values = texture[mask]
    active = values[np.isfinite(values)]
    return {
        "active_vertices": int(len(active)),
        "active_percent": float(100 * len(active) / max(mask.sum(), 1)),
        "max_t": float(np.nanmax(active)) if len(active) else np.nan,
        "mean_t": float(np.nanmean(active)) if len(active) else np.nan,
    }


def plot_flatmap(
    texture,
    title,
    out_path,
    selected_regions,
    region_masks,
    region_colors,
    points,
    flat_triangles,
    n_left,
):
    vmax = float(np.nanmax(texture))
    vertex_data = cortex.Vertex(
        texture, "fsaverage", cmap="hot", vmin=0, vmax=vmax
    )
    fig = cortex.quickflat.make_figure(
        vertex_data,
        with_curvature=True,
        with_colorbar=False,
        with_rois=False,
        with_sulci=False,
        with_labels=False,
        curvature_brightness=0.56,
        curvature_contrast=0.24,
        height=760,
    )
    ax = fig.axes[0]

    for region in selected_regions:
        color = region_colors[region]
        mask = region_masks[region]
        segments = boundary_segments(mask, points, flat_triangles)
        if len(segments):
            ax.add_collection(
                LineCollection(
                    segments,
                    colors=[color],
                    linewidths=2.15,
                    alpha=0.98,
                    zorder=8,
                    capstyle="round",
                    joinstyle="round",
                )
            )

        label = SHORT_LABELS.get(region, region)
        left_mask = mask.copy()
        left_mask[n_left:] = False
        right_mask = mask.copy()
        right_mask[:n_left] = False
        for hemi_mask in (left_mask, right_mask):
            if hemi_mask.sum() < 8:
                continue
            x, y = label_anchor(hemi_mask, points)
            ax.text(
                x,
                y,
                label,
                color="white",
                fontsize=11,
                fontweight="bold",
                ha="center",
                va="center",
                zorder=20,
                path_effects=[pe.withStroke(linewidth=3.2, foreground="black")],
            )

    ax.set_xlim(*FLATMAP_XLIM)
    ax.set_title(title, fontsize=23, fontweight="bold", pad=22)
    brain_position = ax.get_position()
    cbar_ax = fig.add_axes(
        [
            brain_position.x0 + 0.12,
            brain_position.y0 - 0.040,
            brain_position.width - 0.24,
            0.022,
        ]
    )
    scalar_map = cm.ScalarMappable(
        cmap="hot", norm=mcolors.Normalize(vmin=0, vmax=vmax)
    )
    scalar_map.set_array([])
    colorbar = fig.colorbar(scalar_map, cax=cbar_ax, orientation="horizontal")
    colorbar.set_label("t-value", fontsize=18, fontweight="bold", labelpad=4)
    colorbar.ax.tick_params(labelsize=11, width=0.8, length=3)
    colorbar.outline.set_linewidth(0.8)

    fig.savefig(out_path, dpi=240, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved {out_path}")


def combine_images_horizontally(image_paths, out_path):
    images = [Image.open(path).convert("RGB") for path in image_paths]
    height = max(image.height for image in images)

    def centered(image):
        if image.height == height:
            return image
        canvas = Image.new("RGB", (image.width, height), "white")
        canvas.paste(image, (0, (height - image.height) // 2))
        return canvas

    images = [centered(image) for image in images]
    gap = 22
    combined_width = sum(image.width for image in images) + gap * (len(images) - 1)
    combined = Image.new("RGB", (combined_width, height), "white")
    x = 0
    for image in images:
        combined.paste(image, (x, 0))
        x += image.width + gap
    combined.save(out_path, dpi=(240, 240))
    print(f"Saved {out_path}")


def main():
    plt.close("all")
    cortex.database.db = cortex.database.Database()
    PY_CORTEX_DIR.mkdir(parents=True, exist_ok=True)

    fsaverage = datasets.fetch_surf_fsaverage("fsaverage")
    atlas = datasets.fetch_atlas_harvard_oxford(
        ATLAS_NAME, symmetric_split=True, verbose=1
    )

    points, flat_triangles = cortex.db.get_surf(
        "fsaverage", "flat", merge=True, nudge=True
    )
    _, fiducial_triangles = cortex.db.get_surf(
        "fsaverage", "fiducial", merge=True, nudge=True
    )
    n_left = cortex.db.get_surf("fsaverage", "flat", "lh")[0].shape[0]

    print("Projecting Oxford-Harvard atlas to fsaverage...")
    atlas_labels = project_atlas(atlas.maps, fsaverage)
    if len(atlas_labels) != len(points):
        raise RuntimeError(
            f"Atlas has {len(atlas_labels)} vertices, flatmap has {len(points)}."
        )

    grouped_ids = {}
    for atlas_id, label in enumerate(atlas.labels):
        if atlas_id == 0 or label == "Background":
            continue
        grouped_ids.setdefault(base_region(label), []).append(atlas_id)

    raw_masks = {
        region: np.isin(atlas_labels, atlas_ids)
        for region, atlas_ids in grouped_ids.items()
    }

    print("Projecting statistical maps...")
    clip_texture = load_tmap_texture(CLIP_TMAP, fsaverage)
    gist_texture = load_tmap_texture(GIST_TMAP, fsaverage)
    va_texture = load_tmap_texture(VA_TMAP, fsaverage)

    rows = []
    for region, mask in raw_masks.items():
        clip = active_stats(clip_texture, mask)
        gist = active_stats(gist_texture, mask)
        va = active_stats(va_texture, mask)
        rows.append(
            {
                "atlas": "Harvard-Oxford cortical maxprob 25% (1 mm)",
                "region": region,
                "short_label": SHORT_LABELS.get(region, region),
                "atlas_vertices": int(mask.sum()),
                **{f"clip_{key}": value for key, value in clip.items()},
                **{f"gist_{key}": value for key, value in gist.items()},
                **{f"va_{key}": value for key, value in va.items()},
            }
        )

    summary = pd.DataFrame(rows)
    summary["display_score"] = summary[
        ["clip_active_vertices", "gist_active_vertices", "va_active_vertices"]
    ].max(axis=1)
    summary["display_percent"] = summary[
        ["clip_active_percent", "gist_active_percent", "va_active_percent"]
    ].max(axis=1)
    eligible = summary[
        (summary["display_score"] >= MIN_ACTIVE_VERTICES)
        & (summary["display_percent"] >= MIN_ACTIVE_PERCENT)
    ].sort_values(
        ["display_score", "display_percent"], ascending=False
    )
    # Superset/union: show every qualifying parcel active in any of the maps.
    selected_regions = eligible["region"].tolist()
    summary["displayed"] = summary["region"].isin(selected_regions)
    summary.sort_values(
        ["displayed", "display_score"], ascending=False
    ).to_csv(OUT_SUMMARY, index=False)

    print("\nOxford-Harvard parcel union displayed in all three panels:")
    for region in selected_regions:
        print(f"  {SHORT_LABELS.get(region, region):12s}  {region}")

    print("\nCleaning atlas borders on the surface mesh...")
    adjacency = normalized_mesh_adjacency(len(points), fiducial_triangles)
    region_masks = {
        region: smooth_mask(raw_masks[region], adjacency)
        for region in selected_regions
    }
    region_colors = {
        region: ROI_COLORS[i % len(ROI_COLORS)]
        for i, region in enumerate(selected_regions)
    }

    plot_flatmap(
        clip_texture,
        "CLIP-ViT semantic RDM",
        OUT_CLIP,
        selected_regions,
        region_masks,
        region_colors,
        points,
        flat_triangles,
        n_left,
    )
    plot_flatmap(
        gist_texture,
        "GIST low-level RDM",
        OUT_GIST,
        selected_regions,
        region_masks,
        region_colors,
        points,
        flat_triangles,
        n_left,
    )
    plot_flatmap(
        va_texture,
        "Valence-Arousal RDM",
        OUT_VA,
        selected_regions,
        region_masks,
        region_colors,
        points,
        flat_triangles,
        n_left,
    )
    combine_images_horizontally(
        [OUT_GIST, OUT_CLIP, OUT_VA], OUT_COMBINED
    )
    # Keep the generic output name pointing to the newest complete figure.
    Image.open(OUT_COMBINED).save(OUT_LATEST, dpi=(240, 240))
    print(f"Saved {OUT_LATEST}")


if __name__ == "__main__":
    main()


## Rendered output

![Three Oxford-Harvard flatmaps](surface_plot_oxford_harvard_three_maps.png)